<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-09-multimodal-and-pretrained/lesson-9.3-pretrained-apis/notebooks/GCP_Capstone_9.3_PretrainedAPIs.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9.3 Pre-trained APIs — Vision, Natural Language, Translation and Speech, on the Corpus's Real Inputs
**Netsetos GenAI Engineering — GCP Capstone** · Module 9 · rebuilt on the live lane, 9 September 2026

The pre-trained APIs, each on something the corpus actually holds. Vision's OCR of a real Act page beside Document AI's text of the same page; entities in the Code on Wages' own definition of wages; the lane's cited answer translated into three Indian languages and the Hinglish invoice run through language detection; Chirp 3 HD saying the answer and Chirp 3 hearing it back, with a word error rate; retry in the google-cloud clients' own exceptions; a cost comparison on the corpus's real page counts; and the Indian-language strategy on the invoice image. Three of these APIs were not enabled on the lane until this lesson joined it (`make apis`).


## Setup


In [ ]:
!pip install -q google-cloud-vision==3.15.0 google-cloud-language==2.21.0 google-cloud-translate==3.27.0 google-cloud-speech==2.40.0 google-cloud-texttospeech==2.37.0 google-cloud-storage==3.13.1 google-genai==2.22.0 Pillow==12.3.0 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "feat/lesson-4.8-live-evals"        # the demo branch; main is behind it

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson

print("kit:", KIT, "| API:", API_URL, "| media:", f"gs://{MEDIA_BUCKET}")


## Cell 1: The corpus's media


In [ ]:
import json, requests, time
from google.cloud import storage

# THE ROUTES 9.4's Studio stands on, called the way the UI calls them: one ID token per request,
# minted AS the roster member, audience = the API (7.3's hour-long fuse never arms). The body names
# the tenant; the API checks the caller's email on that tenant's roster before it spends a paisa.
def api(path: str, body: dict | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route as documind-ui-sa. Returns (status, json-or-text) - never raises on 4xx,
    because a refusal is data this module reads (the outsider cells)."""
    r = requests.post(f"{API_URL}{path}", json=body,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

# The corpus's media objects, read as YOU (the Colab credential is a project owner; the lane's
# services read them as their own accounts). One client, both buckets.
gcs = storage.Client(project=PROJECT_ID)

def media_objects(prefix: str = f"{TENANT}/") -> list[str]:
    """Every image, video or recording under the tenant's prefix of the uploads bucket."""
    return sorted(b.name for b in gcs.list_blobs(UPLOAD_BUCKET, prefix=prefix)
                  if b.name.lower().endswith((".png", ".jpg", ".jpeg", ".mp4", ".mp3")))

def gcs_bytes(uri: str) -> bytes:
    bucket, _, name = uri.removeprefix("gs://").partition("/")
    return gcs.bucket(bucket).blob(name).download_as_bytes()

FIG3  = f"gs://{UPLOAD_BUCKET}/{TENANT}/annual_report_2026_fig3.png"
INV   = f"gs://{UPLOAD_BUCKET}/{TENANT}/inv_2026_0412.png"
PAGE  = f"gs://{UPLOAD_BUCKET}/{TENANT}/payment_of_bonus_act_1965_p30.png"
VIDEO = f"gs://{UPLOAD_BUCKET}/{TENANT}/townhall_2026_q1.mp4"
POSH  = f"gs://{UPLOAD_BUCKET}/{TENANT}/posh_act_2013.pdf"

have = media_objects()
HAS_VIDEO = f"{TENANT}/townhall_2026_q1.mp4" in have
print("media in the corpus:", have or "NONE - run `make media` and `make ingest-corpus` from deploy/ (README: media is a document)")
print("video:", "present" if HAS_VIDEO else "absent (make media MEDIA_ARGS=--video, or drop a recording in) - the video cells will say so")
assert have, "no media under the tenant's prefix: nothing in this lesson can cite a figure until the corpus holds one"


## Cell 2: Vision OCR of a real page, beside Document AI's
Page 30 of the Payment of Bonus Act - the Fourth Schedule's table - read by Vision, and the same page as the lane holds it: Document AI's chunk and the worker's caption. Three readings of one page.


In [ ]:
from google.cloud import vision
vision_client = vision.ImageAnnotatorClient()

# OCR OF A REAL PAGE, TWO ENGINES. Vision's DOCUMENT_TEXT_DETECTION on the page image the corpus holds
# (page 30 of the Payment of Bonus Act 1965, rendered by make media: the Fourth Schedule's set-on /
# set-off table), beside the text the lane already has for the SAME page - Document AI at ingest (4.1),
# retrievable as a chunk. Same page, two OCRs, one overlap number.
image = vision.Image(source=vision.ImageSource(gcs_image_uri=PAGE))
resp = vision_client.document_text_detection(image=image)
ocr = resp.full_text_annotation.text
blocks = sum(len(pg.blocks) for pg in resp.full_text_annotation.pages)
print(f"Vision: {len(ocr)} chars in {blocks} blocks (Page -> Block -> Paragraph -> Word -> Symbol)")
print(ocr[:280].replace("\n", " | "))
assert "SCHEDULE" in ocr.upper() and "set on" in ocr.lower(), "Vision did not read the schedule page"

hit = documind_tools.retrieve("the Fourth Schedule illustration of set on and set off of allocable surplus",
                              tenant_id=TENANT, top_k=8, brain="direct")
tok = lambda s: {w for w in "".join(ch.lower() if ch.isalnum() else " " for ch in s).split() if len(w) > 3}
page_chunk = next((c for c in hit.get("citations", []) if c["source_uri"].endswith("payment_of_bonus_act_1965.pdf")), None)
figure = next((c for c in hit.get("citations", []) if c.get("kind") == "figure" and c["source_uri"].endswith("p30.png")), None)
if page_chunk:
    overlap = len(tok(ocr) & tok(page_chunk["quote"])) / max(1, len(tok(page_chunk["quote"])))
    print(f"\nDocument AI's chunk (p.{page_chunk.get('page')}): {page_chunk['quote'][:160]!r} ...")
    print(f"words of that chunk that Vision also read: {overlap:.0%}")
if figure:
    print(f"\nand the worker's CAPTION of the same page, a third reading: {figure['quote'][:160]!r} ...")
print("\nkinds the lane returned for this question:", [c.get("kind", "text") for c in hit.get("citations", [])])
assert page_chunk or figure, "the lane returned neither the page's chunk nor the page's figure"

# SafeSearch rides free on any Vision request - the moderation gate 12.6 would put in front of an upload.
ss = vision_client.safe_search_detection(image=image).safe_search_annotation
print("SafeSearch adult / violence:", ss.adult.name, "/", ss.violence.name)


## Cell 3: Vision cost


In [ ]:
# Cost calculator for Vision AI
def vision_cost(monthly_requests, feature='text_detection'):
    free_tier = 1000
    rates = {
        'text_detection': 1.50,
        'document_text_detection': 1.50,
        'label_detection': 1.50,
        'object_localization': 2.25,
        'safe_search_detection': 1.50,
        'face_detection': 1.50,
        'logo_detection': 1.50,
    }
    rate = rates.get(feature, 1.50)
    billable = max(0, monthly_requests - free_tier)
    cost = billable * rate / 1000
    return cost

# DocuMind: 500 invoice images/month
for requests in [100, 500, 1000, 5000, 10000, 50000]:
    cost = vision_cost(requests, 'document_text_detection')
    print(f'{requests:>6,} requests: ${cost:>7.2f}/month')


## Cell 4: Entities in a real clause


In [ ]:
from google.cloud import language_v1 as language
language_client = language.LanguageServiceClient()

# ENTITIES IN A REAL CLAUSE. The text is retrieve()'s: the Code on Wages' own definition of "wages".
# The Natural Language API is English-only for entities; the clause is English, and the invoice
# below is not - which is the whole reason Cell 9 exists.
hit = documind_tools.retrieve("How does the Code on Wages define wages?", tenant_id=TENANT, brain="direct")
clause = next((c["quote"] for c in hit.get("citations", []) if "code_on_wages" in c["source_uri"]), hit.get("answer") or "")
assert clause, hit
doc = language.Document(content=clause, type_=language.Document.Type.PLAIN_TEXT, language="en")
ents = language_client.analyze_entities(document=doc).entities
print(f"{len(ents)} entities in {len(clause)} characters of the Code on Wages:")
for e in sorted(ents, key=lambda e: -e.salience)[:8]:
    print(f"  {e.name[:32]:32} {language.Entity.Type(e.type_).name:14} salience {e.salience:.2f}")
assert ents


In [ ]:
# SEVERAL FEATURES, ONE CALL. annotate_text runs entities, sentiment and syntax together and bills each
# feature separately - one round trip, not three.
features = language.AnnotateTextRequest.Features(extract_entities=True, extract_document_sentiment=True)
ann = language_client.annotate_text(request={"document": doc, "features": features})
print(f"sentiment score {ann.document_sentiment.score:+.2f} magnitude {ann.document_sentiment.magnitude:.2f}",
      "(a statute reads as neutral - which is a useful calibration for the scale)")
print("language:", ann.language, "| entities:", len(ann.entities))


## Cell 5: A real answer, three languages; the invoice, detected


In [ ]:
from google.cloud import translate_v3 as translate
translate_client = translate.TranslationServiceClient()
PARENT = f"projects/{PROJECT_ID}/locations/global"

# A REAL ANSWER, THREE INDIAN LANGUAGES. The source is the lane's cited answer (golden row lk-06), so
# what is translated is a fact the corpus holds. Translation v3 is project-scoped and global.
notice = documind_tools.retrieve("What is the notice period for a confirmed E3?", tenant_id=TENANT, brain="direct")
src = (notice.get("answer") or "")[:400]
assert notice.get("answerable"), notice
print("en:", src[:120], "...")
for lang in ("hi", "ta", "te"):
    t = translate_client.translate_text(parent=PARENT, contents=[src], mime_type="text/plain",
                                        source_language_code="en", target_language_code=lang)
    print(f"{lang}:", t.translations[0].translated_text[:120], "...")

# LANGUAGE DETECTION on the corpus's Hinglish invoice - the register a guard tuned on English misses.
inv = documind_tools.retrieve("What is the GST on invoice INV-2026-0412?", tenant_id=TENANT, brain="direct")
snippet = next((c["quote"] for c in inv.get("citations", []) if "inv_2026_0412" in c["source_uri"]), "")
assert snippet, inv
det = translate_client.detect_language(parent=PARENT, content=snippet[:500], mime_type="text/plain")
print("\nthe invoice snippet, detected:", [(l.language_code, round(l.confidence, 2)) for l in det.languages[:3]],
      "- code-mixed text confuses a detector, and that is the finding")


## Cell 6: Chirp 3, both directions
Regional, and the region decides the model. The loop tries asia-south1 first, the way the UI's `voice.py` does, and falls back.


In [ ]:
from google.cloud import texttospeech as tts
from google.cloud.speech_v2 import SpeechClient
from google.cloud.speech_v2.types import cloud_speech as cs
from google.api_core.client_options import ClientOptions

# ROUND TRIP: Chirp 3 HD SAYS the answer, Chirp 3 HEARS it back, and the word error rate is the one
# number this cell is for. Speech-to-Text v2 is REGIONAL and the region decides the model: chirp_3 is GA
# in us / eu and Preview in asia-south1 (the UI's SPEECH_REGION, where voice.py falls back to chirp_2 and
# long). This list moves; the Locations API is the current one.
tts_client = tts.TextToSpeechClient()
voice_names = [v.name for v in tts_client.list_voices(language_code="en-IN").voices if "Chirp3-HD" in v.name]
assert voice_names, "no en-IN Chirp 3 HD voice listed"
wav = tts_client.synthesize_speech(
    input=tts.SynthesisInput(text=src),
    voice=tts.VoiceSelectionParams(language_code="en-IN", name=voice_names[0]),
    audio_config=tts.AudioConfig(audio_encoding=tts.AudioEncoding.LINEAR16, sample_rate_hertz=16000)).audio_content
print(f"said {len(src)} chars with {voice_names[0]}: {len(wav) // 1024} KB of 16 kHz PCM")

def transcribe(audio: bytes, region: str, model: str) -> str:
    client = SpeechClient(client_options=ClientOptions(api_endpoint=f"{region}-speech.googleapis.com"))
    cfg = cs.RecognitionConfig(auto_decoding_config=cs.AutoDetectDecodingConfig(), language_codes=["en-IN"], model=model,
                               features=cs.RecognitionFeatures(enable_automatic_punctuation=True))
    resp = client.recognize(request=cs.RecognizeRequest(
        recognizer=f"projects/{PROJECT_ID}/locations/{region}/recognizers/_", config=cfg, content=audio))
    return " ".join(r.alternatives[0].transcript for r in resp.results if r.alternatives).strip()

heard, used = "", None
for region, model in (("asia-south1", "chirp_3"), ("asia-south1", "chirp_2"), ("us", "chirp_3")):
    try:
        heard, used = transcribe(wav, region, model), (region, model)
        break
    except Exception as e:
        print(f"  {region}/{model}: {type(e).__name__}: {str(e)[:100]}")
assert heard, "no region/model pair transcribed - is the Speech-to-Text API enabled (make apis)? read the errors above"

def wer(ref: str, hyp: str) -> float:
    norm = lambda s: [w.strip(".,;:") for w in s.lower().split()]
    r, h = norm(ref), norm(hyp)
    d = [[0] * (len(h) + 1) for _ in range(len(r) + 1)]
    for i in range(len(r) + 1): d[i][0] = i
    for j in range(len(h) + 1): d[0][j] = j
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + (r[i - 1] != h[j - 1]))
    return d[-1][-1] / max(1, len(r))

print("heard with", used, ":", heard[:160], "...")
print(f"WER {wer(src, heard):.0%} | the figure survived: {'60' in heard.replace('sixty', '60')}")
assert wer(src, heard) < 0.5


## Cell 7: Retry, in api_core's exceptions


In [ ]:
from google.api_core import exceptions, retry

# THE GOOGLE-CLOUD CLIENTS RAISE api_core EXCEPTIONS - not genai's errors.APIError (9.1). Same policy,
# different library: retry the platform's failures (503, 429, 500, deadline), never a 4xx of your own.
RETRY_CONFIG = retry.Retry(initial=1.0, maximum=60.0, multiplier=2.0, deadline=300.0,
                           predicate=retry.if_exception_type(exceptions.ServiceUnavailable, exceptions.DeadlineExceeded,
                                                             exceptions.ResourceExhausted, exceptions.InternalServerError))

def safe_ocr(gcs_uri: str) -> str | None:
    img = vision.Image(source=vision.ImageSource(gcs_image_uri=gcs_uri))
    try:
        return vision_client.document_text_detection(image=img, retry=RETRY_CONFIG).full_text_annotation.text
    except exceptions.InvalidArgument as e:          # a 400 is yours: never retried
        print("invalid input:", str(e)[:120])
        return None

text = safe_ocr(INV)
print(f"OCR of the invoice page with retry protection: {len(text or '')} chars")
assert text and "1,84,500" in text, "Vision did not read the total off the invoice page"


## Cell 8: Pipeline cost on real page counts


In [ ]:
# PRE-TRAINED PIPELINE VS GEMINI-ONLY, ON THE CORPUS'S REAL PAGE COUNTS. The manifest in the clone
# lists every real document with its pages; the estimate uses them instead of "3 pages per document".
manifest = json.load(open(f"{KIT}/deploy/evals/manifest.json", encoding="utf-8"))
real = [m for m in manifest if m.get("real") and m["tenant_id"] == TENANT and m.get("pages")]
pages = sum(m["pages"] for m in real)
print(f"{len(real)} real documents under {TENANT}, {pages} pages (largest: {max(real, key=lambda m: m['pages'])['slug']})")

def pretrained(total_pages: int, chars_per_page: int = 2000) -> float:
    vision_usd = max(0, total_pages - 1000) * 1.50 / 1000                 # 1K units free, $1.50 / 1K after
    nl_usd = max(0, total_pages * chars_per_page / 1000 - 5000) * 1.00 / 1000
    trans_usd = max(0, total_pages * chars_per_page - 500_000) * 20 / 1_000_000
    return vision_usd + nl_usd + trans_usd

def gemini_only(total_pages: int) -> float:
    return (total_pages * 258 * 1.50 + (total_pages / 3) * 500 * 7.50) / 1_000_000   # 258 tokens a page in, ~500 out per document

print(f"\n{'corpora':>10} {'pages':>8} {'pre-trained':>12} {'gemini-only':>12}  winner")
for n in (1, 10, 50, 100):
    pt, gm = pretrained(pages * n), gemini_only(pages * n)
    print(f"{n:>10} {pages * n:>8,} ${pt:>11.2f} ${gm:>11.2f}  {'pre-trained' if pt < gm else 'Gemini'}")
print("\nthe free tiers decide the first rows; the per-page price decides the rest. And the lane pays Document AI once (4.1) - neither of these, per question.")


## Cell 9: The Indian-language strategy, on the Hinglish invoice


In [ ]:
# THE INDIAN-LANGUAGE STRATEGY, ON THE HINGLISH INVOICE. Two pipelines over the corpus's own code-mixed
# document: layered (Vision OCR -> Translate -> NL entities, three calls, English-only entities) and
# direct (Vision OCR -> Gemini, two calls, code-mixed handled natively). The image is the invoice page.
from pydantic import BaseModel

class InvoiceEntities(BaseModel):
    organisations: list[str]
    dates: list[str]
    amounts: list[str]
    identifiers: list[str]      # PAN, GSTIN, invoice number

source_text = vision_client.document_text_detection(image=vision.Image(source=vision.ImageSource(gcs_image_uri=INV))).full_text_annotation.text

# v1: layered
english = translate_client.translate_text(parent=PARENT, contents=[source_text], target_language_code="en",
                                          mime_type="text/plain").translations[0].translated_text
v1 = language_client.analyze_entities(document=language.Document(content=english, type_=language.Document.Type.PLAIN_TEXT)).entities
print("v1 layered  :", [(e.name, language.Entity.Type(e.type_).name) for e in v1[:6]])

# v2: direct
v2 = gen.models.generate_content(model="gemini-3.6-flash",
    contents=f"Extract the entities from this invoice text (it may mix Hindi and English):\n\n{source_text}",
    config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=InvoiceEntities)).parsed
print("v2 direct   :", v2)
assert any("INV-2026-0412" in x for x in v2.identifiers) and any("1,84,500" in a for a in v2.amounts), v2
print("\nthe direct path read the invoice number and the total exactly; the layered one paid three calls and lost the code-mixed labels in translation")


## Where this goes
- **9.4** uses Gemini for video segments and diarisation, and says why Chirp 3 still matters when the audio must stay in India.
- **9.6** is where the OCR of a page becomes a citation: the caption is the quote, the page image rides beside it.

## ✅ Lesson 9.3 complete
- ✅ Vision OCR of a real page, with Document AI's chunk and the worker's caption of the same page
- ✅ Entities and sentiment in a real clause; a real answer in Hindi, Tamil and Telugu; the Hinglish invoice detected
- ✅ Chirp 3 HD said it, Chirp 3 heard it, the WER printed; the regional fallback the UI uses
- ✅ Retry in api_core's exceptions, never a 400
- ✅ A cost comparison on the corpus's own page counts
- ✅ Layered versus direct on a code-mixed invoice, and which one read the total
